In [0]:
from pyspark.sql.functions import *

In [0]:
dbutils.widgets.text("catalog_name", "finalproject")

In [0]:
catalog_name = dbutils.widgets.get('catalog_name')

In [0]:
df_compras = spark.table(f'{catalog_name}.silver.compras')

df_detalles = spark.table(f'{catalog_name}.silver.detalles')

In [0]:
df_fact_compras = df_compras\
    .join(df_detalles, 
          df_compras.factura == df_detalles.factura, 
          'inner'
    ).select(df_compras.ventaid,df_compras.factura,df_compras.fecha_orden,df_compras.fecha_envio,df_compras.estado,df_compras.metodo_pago,df_compras.grupo_dias_abierto,df_compras.cliente_id,df_compras.tipo_documento,df_compras.num_documento,df_compras.nombre_cliente,df_compras.tipo_cliente,df_compras.vendedor,df_compras.departamento,df_detalles.detalle_id,df_detalles.tienda,df_detalles.categoria,df_detalles.subcategoria,df_detalles.producto,df_detalles.unidades,df_detalles.subtotal)

In [0]:
df_fact_compras = df_fact_compras.withColumn('fecha_carga',current_timestamp())

In [0]:
producto_df = df_fact_compras.select('producto','categoria','subcategoria').dropDuplicates()
cliente_df = df_fact_compras.select('cliente_id','tipo_documento','num_documento','tipo_cliente','nombre_cliente').dropDuplicates()
df_tienda = df_fact_compras.select('tienda').dropDuplicates()
df_compras = df_fact_compras.drop('tipo_documento','num_documento','tipo_cliente','nombre_cliente','categoria','subcategoria')

In [0]:
try:
    df_fact_compras.write.mode('overwrite').format('delta').option('overwriteSchema',True).saveAsTable(f'{catalog_name}.gold.fact_compras')
    df_compras.write.mode('overwrite').format('delta').option('overwriteSchema',True).saveAsTable(f'{catalog_name}.gold.gold_fact_compras')
    producto_df.write.mode('overwrite').format('delta').option('overwriteSchema',True).saveAsTable(f'{catalog_name}.gold.gold_producto')
    cliente_df.write.mode('overwrite').format('delta').option('overwriteSchema',True).saveAsTable(f'{catalog_name}.gold.gold_cliente')
    df_tienda.write.mode('overwrite').format('delta').option('overwriteSchema',True).saveAsTable(f'{catalog_name}.gold.gold_tienda')
    
except Exception as e:
    import traceback
    # Tipo de error
    error_type = type(e).__name__
    # Descripcion del error 
    error_summary = str(e)
    # Traza del error (ver en qué parte se generó el error)
    error_trace = traceback.format_exc()

    # Error completo
    error_msg_full = f"{error_type}: {error_summary}\n{error_trace}"

    if len(error_msg_full) > 500:
        error_msg = error_msg_full[:500] + "\n[...] ERROR TRUNCADO [...]"
    else:
        error_msg = error_msg_full
    
    dbutils.jobs.taskValues.set(key = "error", value = error_msg)
    raise e

## Actualizamos la tabla watermarks

In [0]:
last_updated_presencial = spark.sql(f"""select max(updated_at) from {catalog_name}.bronze.compras where tipo_compra = 'presencial'""").first()[0]
last_updated_online = spark.sql(f"""select max(updated_at) from {catalog_name}.bronze.compras where tipo_compra = 'online'""").first()[0]

In [0]:
spark.sql(f"""
    UPDATE {catalog_name}.bronze.watermarks
    SET max_fecha = TRY_CAST('{last_updated_presencial}' AS TIMESTAMP)
    WHERE collection = 'presencial'
""")
spark.sql(f"""
    UPDATE {catalog_name}.bronze.watermarks
    SET max_fecha = TRY_CAST('{last_updated_online}' AS TIMESTAMP)
    WHERE collection = 'online'
""")